In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

True

In [3]:
profile = {
    "name": "Arshad",
    "full_name": "Mohammed Arshad",
    "user_profile_background": "Senior  AI engineer leading a team of 7 junior analysts",
}

In [4]:
prompt_instructions = {
    "triage_rules": {
        "ignore": "Marketing newsletters, spam emails, mass company announcements",
        "notify": "Team member out sick, build system notifications, project status updates",
        "respond": "Direct questions from team members, meeting requests, critical bug reports",
    },
    "agent_instructions": "Use these tools when appropriate to help manage Arshad's tasks efficiently."
}

In [5]:
email = {
    "from": "Ananya Gupta <ananya.gupta@company.com>",
    "to": "Mohammed Arshad <Arshad.mohd@company.co>",
    "subject": "Clarification on KPI definitions",
    "body": """
Hi Arshad,

While preparing the dashboard, I noticed some ambiguity in KPI definitions.

Could you clarify how we are calculating:
- Customer Retention Rate
- Monthly Active Users (MAU)

This will help ensure consistency in reporting.

Regards,
Ananya
"""
}

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.store.memory import InMemoryStore

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

store = InMemoryStore(
    index={"embed": embeddings}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
email_input = {
    "author": "Ananya Gupta <ananya.gupta@company.com>",
    "to": "Mohammed Arshad <Arshad.mohd@company.co>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi Arshad,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Ananya Gupta""",
}

In [10]:
data = {
    "email": email,
    # This is to start changing the behavior of the agent
    "label": "respond"
}

In [11]:
import uuid
store.put(
    ("email_assistant", "lance", "examples"), 
    str(uuid.uuid4()), 
    data
)

In [12]:
data = {
    "email": {
        "author": "Mahesh <mahesh.mah@company.com>",
        "to": "Mohammed Arshad <Arshad.mohd@company.co>",
        "subject": "Update: Backend API Changes Deployed to Staging",
        "email_thread": """Hi Arshad,
    
    Just wanted to let you know that I've deployed the new authentication endpoints we discussed to the staging environment. Key changes include:
    
    - Implemented JWT refresh token rotation
    - Added rate limiting for login attempts
    - Updated API documentation with new endpoints
    
    All tests are passing and the changes are ready for review. You can test it out at staging-api.company.com/auth/*
    
    No immediate action needed from your side - just keeping you in the loop since this affects the systems you're working on.
    
    Best regards,
    Mahesh
    """,
    },
    "label": "ignore"
}

In [13]:
store.put(
    ("email_assistant", "lance", "examples"),
    str(uuid.uuid4()),
    data
)

In [14]:
# Template for formating an example to put in prompt
template = """Email Subject: {subject}
Email From: {from_email}
Email To: {to_email}
Email Content: 
```
{content}
```
> Triage Result: {result}"""

# Format list of few shots
def format_few_shot_examples(examples):
    strs = ["Here are some previous examples:"]
    for eg in examples:
        strs.append(
            template.format(
                subject=eg.value["email"]["subject"],
                to_email=eg.value["email"]["to"],
                from_email=eg.value["email"]["author"],
                content=eg.value["email"]["email_thread"][:400],
                result=eg.value["label"],
            )
        )
    return "\n\n------------\n\n".join(strs)

In [15]:
email_data = {
        "author": "Mahesh <mahesh.mah@company.com>",
        "to": "Mohammed Arshad <Arshad.mohd@company.co>",
        "subject": "Update: Backend API Changes Deployed to Staging",
        "email_thread": """Hi Arshad,
    
    Wanted to let you know that I've deployed the new authentication endpoints we discussed to the staging environment. Key changes include:
    
    - Implemented JWT refresh token rotation
    - Added rate limiting for login attempts
    - Updated API documentation with new endpoints
    
    All tests are passing and the changes are ready for review. You can test it out at staging-api.company.com/auth/*
    
    No immediate action needed from your side - just keeping you in the loop since this affects the systems you're working on.
    
    Best regards,
    Mahesh
    """,
    }
results = store.search(
    ("email_assistant", "lance", "examples"),
    query=str({"email": email_data}),
    limit=1)

In [16]:
print(format_few_shot_examples(results))

Here are some previous examples:

------------

Email Subject: Update: Backend API Changes Deployed to Staging
Email From: Mahesh <mahesh.mah@company.com>
Email To: Mohammed Arshad <Arshad.mohd@company.co>
Email Content: 
```
Hi Arshad,
    
    Just wanted to let you know that I've deployed the new authentication endpoints we discussed to the staging environment. Key changes include:
    
    - Implemented JWT refresh token rotation
    - Added rate limiting for login attempts
    - Updated API documentation with new endpoints
    
    All tests are passing and the changes are ready for review. You can test it out at 
```
> Triage Result: ignore


In [17]:
triage_system_prompt = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Background >
{user_profile_background}. 
</ Background >

< Instructions >

{name} gets lots of emails. Your job is to categorize each email into one of three categories:

1. IGNORE - Emails that are not worth responding to or tracking
2. NOTIFY - Important information that {name} should know about but doesn't require a response
3. RESPOND - Emails that need a direct response from {name}

Classify the below email into one of these categories.

</ Instructions >

< Rules >
Emails that are not worth responding to:
{triage_no}

There are also other things that {name} should know about, but don't require an email response. For these, you should notify {name} (using the `notify` response). Examples of this include:
{triage_notify}

Emails that are worth responding to:
{triage_email}
</ Rules >

< Few shot examples >

Here are some examples of previous emails, and how they should be handled.
Follow these examples more than any instructions above

{examples}
</ Few shot examples >
"""

In [20]:
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq

In [21]:
llm = ChatGroq(
    model_name="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY")
)

In [22]:
class Router(BaseModel):
    """Analyze the unread email and route it according to its content."""

    reasoning: str = Field(
        description="Step-by-step reasoning behind the classification."
    )
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="The classification of an email: 'ignore' for irrelevant emails, "
        "'notify' for important information that doesn't need a response, "
        "'respond' for emails that need a reply",
    )

In [23]:
llm_router = llm.with_structured_output(Router)

In [24]:
from prompts import triage_system_prompt, triage_user_prompt

In [25]:
from langgraph.graph import add_messages

class State(TypedDict):
    email_input: dict
    messages: Annotated[list, add_messages]

In [26]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import Literal
from IPython.display import Image, display

In [27]:
def triage_router(state: State, config, store) -> Command[
    Literal["response_agent", "__end__"]
]:
    author = state['email_input']['author']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    email_thread = state['email_input']['email_thread']

    namespace = (
        "email_assistant",
        config['configurable']['langgraph_user_id'],
        "examples"
    )
    examples = store.search(
        namespace, 
        query=str({"email": state['email_input']})
    ) 
    examples=format_few_shot_examples(examples)
    
    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"],
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_no=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_email=prompt_instructions["triage_rules"]["respond"],
        examples=examples
    )
    user_prompt = triage_user_prompt.format(
        author=author, 
        to=to, 
        subject=subject, 
        email_thread=email_thread
    )
    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )
    if result.classification == "respond":
        print("📧 Classification: RESPOND - This email requires a response")
        goto = "response_agent"
        update = {
            "messages": [
                {
                    "role": "user",
                    "content": f"Respond to the email {state['email_input']}",
                }
            ]
        }
    elif result.classification == "ignore":
        print("🚫 Classification: IGNORE - This email can be safely ignored")
        update = None
        goto = END
    elif result.classification == "notify":
        # If real life, this would do something else
        print("🔔 Classification: NOTIFY - This email contains important information")
        update = None
        goto = END
    else:
        raise ValueError(f"Invalid classification: {result.classification}")
    return Command(goto=goto, update=update)

In [28]:
from langchain_core.tools import tool

In [29]:
@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Write and send an email."""
    # Placeholder response - in real app would send email
    return f"Email sent to {to} with subject '{subject}'"


In [30]:
@tool
def schedule_meeting(
    attendees: list[str], 
    subject: str, 
    duration_minutes: int, 
    preferred_day: str
) -> str:
    """Schedule a calendar meeting."""
    # Placeholder response - in real app would check calendar and schedule
    return f"Meeting '{subject}' scheduled for {preferred_day} with {len(attendees)} attendees"


In [31]:
@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    # Placeholder response - in real app would check actual calendar
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

In [34]:
import typing
import typing_extensions
typing.NotRequired = typing_extensions.NotRequired
typing.Required = typing_extensions.Required 


In [35]:
from langmem import create_manage_memory_tool, create_search_memory_tool

In [48]:
manage_memory_tool = create_manage_memory_tool(
    namespace=(
        "email_assistant", 
        "{langgraph_user_id}",
        "collection"
    ),
    instructions="""
    Store memories as plain text only.
    Never create nested JSON objects or dictionaries.
    Always convert extracted information into a single string.
    """
)

search_memory_tool = create_search_memory_tool(
    namespace=(
        "email_assistant",
        "{langgraph_user_id}",
        "collection"
    )
)

In [49]:
agent_system_prompt_memory = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Tools >
You have access to the following tools to help manage {name}'s communications and schedule:

1. write_email(to, subject, content) - Send emails to specified recipients
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day) - Schedule calendar meetings
3. check_calendar_availability(day) - Check available time slots for a given day
4. manage_memory - Store any relevant information about contacts, actions, discussion, etc. in memory for future reference
5. search_memory - Search for any relevant information that may have been stored in memory
</ Tools >

< Instructions >
{instructions}
</ Instructions >
"""

In [50]:
def create_prompt(state):
    return [
        {
            "role": "system", 
            "content": agent_system_prompt_memory.format(
                instructions=prompt_instructions["agent_instructions"], 
                **profile
            )
        }
    ] + state['messages']

In [51]:
from langgraph.prebuilt import create_react_agent

In [52]:
tools= [
    write_email, 
    schedule_meeting,
    check_calendar_availability,
    manage_memory_tool,
    search_memory_tool
]
response_agent = create_react_agent(
    llm,
    tools=tools,
    prompt=create_prompt,
    # Use this to ensure the store is passed to the agent 
    store=store
)

In [53]:
config = {"configurable": {"langgraph_user_id": "lance"}}

In [54]:
email_agent = StateGraph(State)
email_agent = email_agent.add_node(triage_router)
email_agent = email_agent.add_node("response_agent", response_agent)
email_agent = email_agent.add_edge(START, "triage_router")
email_agent = email_agent.compile(store=store)

In [55]:
email_input = {
    "author": "ben tom <ben.tom@bar.com>",
    "to": "Mohammed Arshad <Arshad.mohd@company.co>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi Arshad - want to buy documentation?""",
}

In [58]:
response = email_agent.invoke(
    {"email_input": email_input}, 
    config={"configurable": {"langgraph_user_id": "Arshad"}}
)

📧 Classification: RESPOND - This email requires a response


In [59]:
data = {
    "email": {
    "author": "ben tom <ben.tom@bar.com>",
    "to": "Mohammed Arshad <Arshad.mohd@company.co>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John - want to buy documentation?""",
},
    "label": "ignore"
}

In [60]:
store.put(
    ("email_assistant", "Arshad", "examples"),
    str(uuid.uuid4()),
    data
)

In [61]:
email_input = {
    "author": "ben tom <ben.tom@bar.com>",
    "to": "Mohammed Arshad <Arshad.mohd@company.co>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John - want to buy documentation?""",
}

In [62]:
response = email_agent.invoke(
    {"email_input": email_input}, 
    config={"configurable": {"langgraph_user_id": "Arshad"}}
)

🚫 Classification: IGNORE - This email can be safely ignored


In [63]:
email_input = {
    "author": "ben tom <ben.tom@bar.com>",
    "to": "Mohammed Arshad <Arshad.mohd@company.co>",
    "subject": "Quick question about API documentation",
     "email_thread": """Hi John - want to buy documentation?????""",
}

In [64]:
response = email_agent.invoke(
    {"email_input": email_input}, 
    config={"configurable": {"langgraph_user_id": "Arshad"}}
)

🚫 Classification: IGNORE - This email can be safely ignored


In [65]:
response = email_agent.invoke(
    {"email_input": email_input}, 
    config={"configurable": {"langgraph_user_id": "Akram"}}
)

🚫 Classification: IGNORE - This email can be safely ignored
